# SatQuery Phase 2B — Kaggle runner
This notebook is intentionally thin. Dataset preparation, preprocessing, LoRA training, checkpointing, and evaluation remain in the repository. Enable a Kaggle GPU and Internet before running all cells.

In [ ]:
import subprocess
import sys

# Do this before importing torch. Kaggle's torchao build is incompatible with
# the verified P100 PyTorch/CUDA stack used by this runner.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
    'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu126',
], check=True)

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

import torch

assert torch.__version__ == '2.8.0+cu126', torch.__version__
assert torch.version.cuda == '12.6', torch.version.cuda
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings → Accelerator.'
gpu = torch.cuda.get_device_properties(0)
print({
    'python': platform.python_version(),
    'pytorch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu': gpu.name,
    'vram_gib': round(gpu.total_memory / 1024**3, 2),
    'compute_capability': torch.cuda.get_device_capability(0),
    'bf16_runtime_reported': torch.cuda.is_bf16_supported(),
})

In [ ]:
REPO_URL = os.environ.get('SATQUERY_REPO_URL', 'https://github.com/bishuk-dev/SIH-26167-SATQuery.git')
REPO_DIR = Path('/kaggle/working/SIH-26167-SATQuery')
OUTPUT_DIR = Path('/kaggle/working/satquery-output/phase2b-smoke')
# Optional attached Kaggle Dataset directory containing the manifest image files.
ATTACHED_DATA_ROOT = os.environ.get('SATQUERY_DATA_ROOT')
os.environ['MODEL_ROOT'] = '/kaggle/temp/satquery-models'

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{REPO_DIR}[training]'], check=True)

In [ ]:
if ATTACHED_DATA_ROOT is None:
    subprocess.run(
        [sys.executable, '-m', 'ml.evaluation.prepare_phase2b_rsvqa'],
        cwd=REPO_DIR,
        check=True,
    )
else:
    print(f'Using attached read-only data from {ATTACHED_DATA_ROOT}')

In [ ]:
command = [
    sys.executable, '-m', 'ml.training.phase2b',
    '--config', 'ml/configs/phase2b_smolvlm_lora.yaml',
    '--output-dir', str(OUTPUT_DIR),
    '--smoke-test',
]
if ATTACHED_DATA_ROOT:
    command.extend(['--data-root', ATTACHED_DATA_ROOT])
subprocess.run(command, cwd=REPO_DIR, check=True)

## Full run (intentionally not started here)
After the one-step smoke run succeeds, follow `docs/KAGGLE.md` to run Phase 2B deliberately. Do not tune using the test comparison.

In [ ]:
print('Smoke artifacts:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size)